# Kalshi favourite-longshot bias: replication, extension and favourite buying

Module 3, hedge fund strategies. Plan: [PLAN.md](PLAN.md).

1. Replicate Bürgi, Deng and Whelan, *Makers and Takers: The Economics of the Kalshi Prediction Market*, on their sample (2021 to April 2025).
2. Extend the same tables out of sample (May 2025 to October 2026).
3. Backtest a weekly favourite-buying portfolio (85–97c), then improve it with a calibration model.

This notebook does the data cleaning and evaluation. The data comes from `download_data.py`, which saves it under `data/`:

```
.venv/bin/python download_data.py
```

The notebook never calls the API, so it runs top to bottom in seconds once the data is on disk.

## 1. Setup

In [1]:
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

In [2]:
DATA = Path("data")

# --- Sample dates ---
PAPER_START = dt.datetime(2021, 1, 1, tzinfo=dt.timezone.utc)
PAPER_END   = dt.datetime(2025, 5, 1, tzinfo=dt.timezone.utc)    # paper sample ends April 2025
OOS_END     = dt.datetime(2026, 11, 1, tzinfo=dt.timezone.utc)   # out of sample: May 2025 to 31 Oct 2026

# --- Paper sample filters (section 4) ---
MIN_VOLUME_DOLLARS = 1_000
MAX_FINAL_SPREAD = 0.20
MIN_HOURS_OPEN = 24

# --- Strategy (section 7) ---
TAKER_FEE_RATE = 0.07      # fee = 0.07 * p * (1 - p) per contract, rounded up to the cent
INTEREST_RATE = 0.035      # annual interest on cash and positions
FAVOURITE_MIN, FAVOURITE_MAX = 0.85, 0.97
MIN_DAYS_TO_CLOSE, MAX_DAYS_TO_CLOSE = 1, 21
EVENT_CAP = 0.03

## 2. Load data

Three files written by `download_data.py`. Values are stored as text, exactly as the API returned them; section 4 converts them.

- `series.parquet`: every Kalshi series with its category and frequency.
- `markets.parquet`: every settled market in the series we keep (sports, hourly, 15-minute and combo series are left out).
- `candles.parquet`: daily price, bid, ask and volume for markets with at least 1,000 contracts traded that were open at least 24 hours.

In [3]:
series = pl.read_parquet(DATA / "series.parquet")
markets = pl.read_parquet(DATA / "markets.parquet")
candles = pl.read_parquet(DATA / "candles.parquet")

print(f"{len(series):,} series, {len(markets):,} settled markets, "
      f"{len(candles):,} daily candles for {candles['ticker'].n_unique():,} markets")

16,191 series, 681,951 settled markets, 3,654,660 daily candles for 226,624 markets


Markets with candles, by category and the year they closed.

In [4]:
with_candles = (
    markets
    .filter(pl.col("ticker").is_in(candles["ticker"].unique()))
    .join(series.select(pl.col("ticker").alias("series_ticker"), "category"), on="series_ticker", how="left")
    .with_columns(pl.col("close_time").str.to_datetime(time_zone="UTC").dt.year().alias("year"))
)
(with_candles
 .group_by("category", "year").len()
 .pivot(on="year", index="category", values="len", sort_columns=True)
 .sort("category"))

/var/folders/wm/nxkcxnfx12v9kprllw20m8qh0000gn/T/ipykernel_96618/3187894488.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter(pl.col("ticker").is_in(candles["ticker"].unique()))


category,2021,2022,2023,2024,2025,2026
str,u32,u32,u32,u32,u32,u32
"""Climate and Weather""",518,2524,2026,5616,13894,54363
"""Commodities""",null,146,123,71,44,15188
"""Companies""",null,null,6,10,39,81
"""Crypto""",null,null,null,180,382,1624
"""Culture""",null,null,null,null,null,1
…,…,…,…,…,…,…
"""Politics""",52,649,445,566,2467,6353
"""Science and Technology""",13,8,1,103,390,4159
"""Social""",null,null,null,null,38,9


## 3. Helper functions

Fee, settlement portfolio, calibration table and performance statistics. To be written with sections 5 and 7.

In [5]:
def taker_fee(price, contracts=1):
    """Kalshi taker fee in dollars: 0.07 * contracts * p * (1 - p), rounded up to the cent."""
    return np.ceil(TAKER_FEE_RATE * contracts * price * (1 - price) * 100 - 1e-9) / 100

## 4. Paper sample

*To do:* Convert types, apply the paper's filters, one row per contract side (Yes and No) per snapshot, count check against the paper's 46,282 contracts and 12,403 events. Save the sample tickers to `data/trade_tickers.parquet` so `download_data.py trades` can fetch their trades.

## 5. Replication

*To do:* Return by 10c price bucket before and after fees, price regression, makers vs takers, by category and year.

## 6. Out of sample

*To do:* The same tables for May 2025 to October 2026.

## 7. Strategy backtest

*To do:* Weekly favourite buying, 85–97c, 1–21 days to close; NAV, performance and tail statistics.

## 8. Calibration model

*To do:* Logistic regression fit on pre-May-2025 data, reliability check, out-of-sample comparison with section 7.

## 9. Conclusions

*To do:* Findings and judgment.